## Exploring EPMT DB

This notebook is trying to explore beyond the 'EPMT_JOB_TAGS' within 'annotations' for rows within the EPMT database.

In [1]:
# Environment work around to get matplotlib/seaborn
import sys
sys.path.append('/home/Janice.Kim/work/epmt/misc/jk_py37')

In [2]:
import epmt_query as eq
from pprint import pprint
from datetime import datetime
from pathlib import Path
import csv
import pandas as pd

In [3]:
# Query how many rows exist
eq.get_jobs(fmt='orm').count()

53529

In [4]:
one=eq.get_jobs(fmt='dict', limit=1)[0]
pprint(one)

{'all_proc_tags': [{'op': 'dmput', 'op_instance': '1'},
                   {'op': 'dmput', 'op_instance': '2'},
                   {'op': 'dmput', 'op_instance': '3'},
                   {'op': 'hsmget', 'op_instance': '1'},
                   {'op': 'hsmget', 'op_instance': '3'},
                   {'op': 'hsmget', 'op_instance': '4'},
                   {'op': 'hsmget', 'op_instance': '6'},
                   {'op': 'hsmget', 'op_instance': '7'},
                   {'op': 'mv', 'op_instance': '1'},
                   {'op': 'mv', 'op_instance': '2'},
                   {'op': 'mv', 'op_instance': '4'},
                   {'op': 'mv', 'op_instance': '6'},
                   {'op': 'mv', 'op_instance': '7'},
                   {'op': 'mv', 'op_instance': '9'},
                   {'op': 'ncatted', 'op_instance': '1'},
                   {'op': 'ncatted', 'op_instance': '3'},
                   {'op': 'ncatted', 'op_instance': '5'},
                   {'op': 'ncks', 'op_instance': '1'},


In [5]:
# These look potentially useful...
print(one['env_dict']['SLURM_MEM_PER_CPU'])
print(one['env_dict']['SLURM_MEM_PER_NODE'])

16384
204800


In [6]:
jobs=eq.get_jobs(fmt='dict', limit=100)

In [7]:
# Oh no... they're possibly the same for all the jobs?
jobs[:]
mem_per_cpu = set()
mem_per_node = set()
loaded_modules = set()
mem_per_node_missing_count = 0

for job in jobs:
    mem_per_cpu.add(job['env_dict']['SLURM_MEM_PER_CPU'])
    if 'SLURM_MEM_PER_NODE' in job['env_dict']:
        mem_per_node.add(job['env_dict']['SLURM_MEM_PER_NODE'])
    else:
        mem_per_node_missing_count += 1
    loaded_modules.add(job['env_dict']['LOADEDMODULES'])
print(mem_per_cpu, mem_per_node, loaded_modules, mem_per_node_missing_count)

{'16384'} {'204800'} {'intel-oneapi-compilers/2024.1.0:zlib-ng/2.1.6:libpng/1.6.39:hdf5/1.14.3:netcdf-c/4.9.2:udunits/2.2.28:ncview/2.1.9:gsl/2.8:nco/5.2.4:python/3.11:epmt/4.9.2', 'epmt/4.9.2'} 82


## More about the data

See the following link for more description about the data: https://gitlab.com/minimal-metrics-llc/epmt/epmt#performance-metrics-data-dictionary

There are more features to explore or predict. For example:
* 41. read_bytes Thread Bytes read from I/O device 
* 42. write_bytes Thread Bytes written to I/O device
* 45. time_waiting Thread Nanoseconds runnable but waiting


In [8]:
# Just some that look potentially interesting...
print(one['read_bytes'])
print(one['write_bytes'])
print(one['time_waiting'])
print(one['cpu_time'])
print(one['duration'])
print(one['start'])
print(one['end'])
print(one['minflt'])
print(one['majflt'])
print(one['tags'])
print(one['annotations'])
print(one['env_dict']['SLURM_JOB_ACCOUNT'])
print(one['env_dict']['SLURM_NTASKS'])
print(one['env_dict']['SLURM_TASKS_PER_NODE'])
print(one['env_dict']['SLURM_MEM_PER_CPU'])
print(one['env_dict']['SLURM_MEM_PER_NODE'])
print(one['env_dict']['SLURM_SCRIPT_CONTEXT'])
print(one['env_dict']['LOADEDMODULES'])
print(one['all_proc_tags']) # Maybe features?
print(one['exitcode']) # Maybe prune out the ones that aren't 0?

16191488
6777675776
6623590945
103951272.0
472841476.0
2026-01-03 04:54:01.967062
2026-01-03 05:01:54.808538
13333426
63
{'exp_name': 'SPEAR_c384_OM4p08_Control_1990_A13', 'exp_time': '01000101', 'exp_target': 'repro-openmp', 'exp_fre_mod': '/home/fms/local/opt/fre-commands/bronx-23', 'script_name': 'SPEAR_c384_OM4p08_Control_1990_A13_river_cubic_01000101', 'exp_platform': 'gfdl.ncrc5-intel23_2_0', 'exp_component': 'river_cubic', 'exp_seg_months': '12'}
{'EPMT_JOB_TAGS': 'exp_component:river_cubic;exp_fre_mod:/home/fms/local/opt/fre-commands/bronx-23;exp_name:SPEAR_c384_OM4p08_Control_1990_A13;exp_time:01000101;exp_platform:gfdl.ncrc5-intel23_2_0;exp_target:repro-openmp;exp_seg_months:12;script_name:SPEAR_c384_OM4p08_Control_1990_A13_river_cubic_01000101'}
gfdl_sd
1
1
16384
204800
prolog_task
epmt/4.9.2
[{'op': 'dmput', 'op_instance': '1'}, {'op': 'dmput', 'op_instance': '2'}, {'op': 'dmput', 'op_instance': '3'}, {'op': 'hsmget', 'op_instance': '1'}, {'op': 'hsmget', 'op_instance': '3'

## Let's collect all of these for 40k rows

I'm not sure the best way to do this. I'll try just to save the data as is into a csv file. Once we have it, we can load it into memory and try to clean it up later.

In [9]:
nrows = 20000 # TODO: increase limit once things look okay, but 50k may be too large at one time?
all_jobs = eq.get_jobs(fmt='dict', limit=nrows) 

keys = ['read_bytes', 'write_bytes', 'time_waiting','cpu_time', 'duration', 'start', 'end', 'minflt', 'majflt', 'tags', 'annotations', 'all_proc_tags', 'exitcode']
env_dict_keys = ['SLURM_JOB_ACCOUNT', 'SLURM_NTASKS', 'SLURM_TASKS_PER_NODE', 'SLURM_MEM_PER_CPU', 'SLURM_MEM_PER_NODE', 'SLURM_SCRIPT_CONTEXT', 'LOADEDMODULES']

data = []
for job in all_jobs:
    row = {}
    for key in keys:
        if key in job:
            row[key] = job[key]
        else:
            row[key] = ""
    for ed_key in env_dict_keys:
        if ed_key in job['env_dict']:
            row[ed_key] = job['env_dict'][ed_key]
        else:
            row[ed_key] = ""
    data.append(row)

labels = data[0].keys()

In [10]:
# I don't want a massive csv file, so I'm going to chunk it for now. I like csv files
# because I can just open them up and look at them if I want. In the future, maybe 
# something like parquet or feather are better.

total_rows = len(data)
print(f"Processing {total_rows} rows.")

max_rows_per_file = 5000
date_str = datetime.today().strftime("%Y%m%d_%H%M%S")

work_dir = Path(f"/home/Janice.Kim/work/epmt/data_{total_rows}_{date_str}")
work_dir.mkdir(parents=True, exist_ok=True)

for i in range(0, total_rows, max_rows_per_file):
    data_chunk = data[i:i+max_rows_per_file]
    part_num = i // max_rows_per_file + 1
    filepath = work_dir / f"jk_data_{total_rows}_{part_num}_{date_str}.csv"
    print(filepath)

    with open(filepath, "w", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=labels)
        writer.writeheader()
        writer.writerows(data_chunk)

Processing 20000 rows.
/home/Janice.Kim/work/epmt/data_20000_20260204_172700/jk_data_20000_1_20260204_172700.csv
/home/Janice.Kim/work/epmt/data_20000_20260204_172700/jk_data_20000_2_20260204_172700.csv
/home/Janice.Kim/work/epmt/data_20000_20260204_172700/jk_data_20000_3_20260204_172700.csv
/home/Janice.Kim/work/epmt/data_20000_20260204_172700/jk_data_20000_4_20260204_172700.csv


Now that we have it in a file, can we clean it up?